# Plan and sample

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/basics/03_plan_and_sample.ipynb)

State a task as a planning problem, solve it, compare it with a hand-tuned controller, and run a controller on a digital clock.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. A planning problem in five lines

A `PlanningProblem` states a task instead of a controller: a system, a start, a cost to
minimize and a horizon. Here: bring an inverted pendulum, tilted by 0.4 rad, back to the
upright, with the quadratic cost

$$J = \int_0^{t_f} x^T Q\, x + u^T R\, u \; dt.$$

A planner reads the problem and returns a solution. `LQRPlanner` linearizes at the target
and solves the Riccati equation; on this finite horizon it is the differential one, swept
back from $t_f$, and its solution is a time-varying feedback policy $u = \pi(x, t) = -K(t)\,x$.
`solve(evaluate=True)` also rolls the policy out from the start and scores it.

In [ ]:
import numpy as np

from minilink import InvertedPendulum, LQRPlanner, PlanningProblem, QuadraticCost

plant = InvertedPendulum()
cost = QuadraticCost.from_system(plant, Q=np.diag([10.0, 1.0]), R=0.1 * np.eye(1))
problem = PlanningProblem(plant, x0=np.array([0.4, 0.0]), cost=cost, horizon=4.0)
lqr_solution = LQRPlanner(problem).solve(evaluate=True)
lqr_solution

In [ ]:
plant.plot_trajectory(lqr_solution.trajectory)

`compare` scores several answers to the same problem on the same run: here the LQR policy
against a hand-tuned `PD` controller. The table gives each one's cost $J$; the LQR policy
is the optimum for this cost, so the PD law scores higher.

In [ ]:
from minilink import PD, compare

hand_tuned = PD(Kp=20.0, Kd=4.0, tau=0.05)
comparison = compare(problem, LQR=lqr_solution, PD=hand_tuned, dt=0.01)
print(comparison)

## 2. Trajectory optimization: a swing-up

Some tasks are out of reach of a linear law. Swinging a pendulum from the bottom to the
top with a limited torque needs a plan. The problem adds the input set `U` (the motor's
limits, $|u| \le 5$) and the target set `Xf`; `TrajectoryOptimizationPlanner` turns it into
a finite optimization on 25 time samples (direct collocation) and solves it.

In [ ]:
from minilink import Pendulum, TrajectoryOptimizationPlanner

pendulum = Pendulum()
x_up = np.array([np.pi, 0.0])
cost = QuadraticCost.from_system(pendulum, x_bar=x_up, Q=np.eye(2), R=0.1 * np.eye(1))
swing_up = PlanningProblem(
    pendulum, x0=np.zeros(2), cost=cost, U=pendulum.input_box, Xf=x_up, horizon=5.0
)
planner = TrajectoryOptimizationPlanner(
    swing_up, n_steps=25, transcription="direct_collocation"
)
plan = planner.solve()
plan

The plan is a trajectory: the torque swings back and forth within its limits to pump
energy into the pendulum, which arrives upright at $t_f$.

In [ ]:
pendulum.plot_trajectory(plan.trajectory)

In [ ]:
pendulum.animate(plan.trajectory)

## 3. A digital controller on a clock

A real controller runs on a computer: it reads the sensor every $\Delta t$ seconds and holds
its command between ticks. `controller % dt` puts a controller on that clock, and
`(controller % dt) @ plant` samples the continuous plant at each tick and holds the
command (zero-order hold). At 100 Hz the PD law behaves like its continuous version:

In [ ]:
plant.x0 = np.array([0.4, 0.0])
fast_loop = (hand_tuned % 0.01) @ plant
fast_loop.compute_trajectory(tf=4.0)
fast_loop.plot_trajectory()

At 10 Hz the same gains act on stale measurements: the delay costs phase margin, and the
pendulum overshoots and rings before it settles.

In [ ]:
slow_loop = (hand_tuned % 0.1) @ plant
slow_loop.compute_trajectory(tf=4.0)
slow_loop.plot_trajectory()

## Where to go next

The basics end here. Two layers go further:

- [core/](../core/01_systems.ipynb), the foundation and the pipeline: systems and
  parameters, sets, fields and distributions, costs, the planning problem, and how a
  problem compiles into one evaluator that simulates, scores and differentiates it.
- [advanced/](../advanced/), the tools: model predictive control, value iteration and the
  other planners, discrete-time and hybrid loops, and reinforcement learning.